In [ ]:
# Make this notebook work from either fine-tuning/ or fine-tuning/live-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 06 · Date & Time Awareness — stop the guessing (LIVE)

> **LIVE DEMO LAB.** Resources are suffixed `-live` so this run never overwrites the pre-demo lab.

A member asks "when does my mail-order arrive?" — but the model doesn't know today's date, so it guesses delivery windows. This lab makes time deterministic: inject the current date into the system prompt, add a `get_current_time` tool the model can call, and add a Python ETA helper that respects the 3 PM PT mail-order cutoff. The result is an accurate arrival date computed in code, not hallucinated. *The cutoff math is Python's job, not the model's.*

---
## Step 1 — Config & client

*Load `.env` and build the `AzureOpenAI` client we'll reuse for every test.*


In [ ]:
# Standard config + client. Same .env all labs share.
import os, json, time
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

load_dotenv()
AZURE_OPENAI_ENDPOINT    = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_DEPLOYMENT          = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')

_cred = DefaultAzureCredential()
_tp   = get_bearer_token_provider(_cred, 'https://cognitiveservices.azure.com/.default')

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = _tp,
    api_version    = AZURE_OPENAI_API_VERSION,
)
print('Client OK. Deployment:', BASE_DEPLOYMENT)


---
## Step 2 — BEFORE: model with no date context

*Ask *"when will my order arrive?"* with no date context at all — the model has to guess what today is.*


In [ ]:
import datetime as dt
NAIVE_SYS = 'You are a Acme Health voice agent. Be concise.'

q = "I'm placing a mail order today. When will my prescription arrive?"
resp = client.chat.completions.create(
    model=BASE_DEPLOYMENT,
    messages=[{'role':'system','content':NAIVE_SYS},{'role':'user','content':q}],
    temperature=0.0, max_tokens=160,
)
print(resp.choices[0].message.content)
print('\n(Notice the model has to guess what "today" is.)')


---
## Step 3 — Fix 1: inject the date in the system prompt

*Fix 1: Inject the real date into the system prompt. One line of code, no extra round trips.*


In [ ]:
today = dt.datetime.now()
DATED_SYS = (
    'You are a Acme Health voice agent. Be concise.\n'
    f'TODAY IS: {today.strftime("%A, %B %d, %Y, %I:%M %p %Z")} (America/Los_Angeles).\n'
    'When the member asks about delivery, compute from today.'
)
resp = client.chat.completions.create(
    model=BASE_DEPLOYMENT,
    messages=[{'role':'system','content':DATED_SYS},{'role':'user','content':q}],
    temperature=0.0, max_tokens=160,
)
print(resp.choices[0].message.content)


---
## Step 4 — Fix 2: give the model a `get_current_time` tool

*Fix 2: Give the model a `get_current_time` tool. The model now *asks* for the time and we hand it back — perfect for long-running agents.*


In [ ]:
# Same question as before. This time we hand the model a tool: get_current_time.
# We also tell the model the Acme rules so once it knows "now" it can compute a real date.
TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'get_current_time',
            'description': 'Return the current date and time in America/Los_Angeles.',
            'parameters': { 'type': 'object', 'properties': {}, 'required': [] },
        },
    },
]

def get_current_time():
    return { 'now_iso': dt.datetime.now().isoformat(), 'timezone': 'America/Los_Angeles' }

TOOL_SYS = (
    'You are a Acme Health voice agent. Be concise.\n'
    'ACME MAIL-ORDER RULES:\n'
    ' - Mail-order cutoff is 3:00 PM Pacific Time.\n'
    ' - Orders placed at or after 3:00 PM PT ship the NEXT business day.\n'
    ' - Skip weekends.\n'
    ' - Transit time is 3 business days from the ship date.\n'
    'When the answer depends on today\'s date, call get_current_time first, then compute the actual ship date and arrival date.'
)

messages = [
    {'role':'system','content': TOOL_SYS},
    {'role':'user','content': q},
]

print('=' * 70)
print('USER QUESTION:')
print(f'  {q}')
print('=' * 70)

# Turn 1: force the model to call the tool so the demo always shows the round-trip.
r = client.chat.completions.create(
    model=BASE_DEPLOYMENT,
    messages=messages,
    tools=TOOLS,
    tool_choice={'type':'function','function':{'name':'get_current_time'}},
    temperature=0.0,
    max_tokens=200,
)
m = r.choices[0].message

print('\n[1] MODEL REQUESTED A TOOL CALL')
for tc in (m.tool_calls or []):
    print(f'      -> {tc.function.name}({tc.function.arguments or "{}"})')

# Hand the tool result back to the model.
messages.append({'role':'assistant','tool_calls':[tc.model_dump() for tc in m.tool_calls]})
tool_outputs = []
for tc in m.tool_calls:
    result = get_current_time() if tc.function.name == 'get_current_time' else {}
    tool_outputs.append(result)
    messages.append({'role':'tool','tool_call_id':tc.id,'content':json.dumps(result)})

print('\n[2] WE RAN THE TOOL AND RETURNED:')
for out in tool_outputs:
    print(f'      {json.dumps(out)}')

# Turn 2: model now answers with the real time + rules in hand.
r2 = client.chat.completions.create(
    model=BASE_DEPLOYMENT,
    messages=messages,
    tools=TOOLS,
    temperature=0.0,
    max_tokens=250,
)
print('\n[3] AGENT FINAL ANSWER:')
print(f'  {r2.choices[0].message.content}')
print('\n(Watch Step 5 — we replace the model\'s math with a deterministic Python function.)')


---
## Step 5 — Fix 3: Acme business calendar tool

*Fix 3: Wire Acme's actual mail-order rules (3 PM PT cutoff, skip weekends, 3 business-day transit) into a deterministic Python helper the model calls.*


In [ ]:
def acme_mail_order_eta(order_dt=None):
    """Return ship_date and earliest_arrival per Acme policy.
    Cutoff 3pm PT, Mon-Fri. 3-5 business days transit."""
    order_dt = order_dt or dt.datetime.now()
    # cutoff
    cutoff = order_dt.replace(hour=15, minute=0, second=0, microsecond=0)
    ship = order_dt.date() if order_dt < cutoff else order_dt.date() + dt.timedelta(days=1)
    # skip weekends
    while ship.weekday() >= 5:
        ship += dt.timedelta(days=1)
    # 3 business days transit
    arrive = ship
    days = 0
    while days < 3:
        arrive += dt.timedelta(days=1)
        if arrive.weekday() < 5:
            days += 1
    return { 'ship_date': ship.isoformat(), 'earliest_arrival': arrive.isoformat() }

CALENDAR_TOOL = {
    'type': 'function',
    'function': {
        'name': 'acme_mail_order_eta',
        'description': 'Compute the Acme mail-order ship date and earliest arrival, honoring the 3 PM PT cutoff and business days.',
        'parameters': {
            'type': 'object',
            'properties': { 'order_iso': { 'type': 'string', 'description': 'Order timestamp ISO 8601. Defaults to now.' } },
            'required': []
        },
    },
}

messages = [
    {'role':'system','content':'You are a Acme Health voice agent. Use acme_mail_order_eta for any delivery-date question. Today is ' + dt.datetime.now().isoformat()},
    {'role':'user','content': q},
]
r = client.chat.completions.create(model=BASE_DEPLOYMENT, messages=messages, tools=[CALENDAR_TOOL], temperature=0.0, max_tokens=200)
m = r.choices[0].message

if m.tool_calls:
    messages.append({ 'role': 'assistant', 'tool_calls': [tc.model_dump() for tc in m.tool_calls] })
    for tc in m.tool_calls:
        args = json.loads(tc.function.arguments or '{}')
        if tc.function.name == 'acme_mail_order_eta':
            out = acme_mail_order_eta(dt.datetime.fromisoformat(args['order_iso']) if args.get('order_iso') else None)
        else:
            out = {}
        messages.append({ 'role': 'tool', 'tool_call_id': tc.id, 'content': json.dumps(out) })
    r2 = client.chat.completions.create(model=BASE_DEPLOYMENT, messages=messages, tools=[CALENDAR_TOOL], temperature=0.0, max_tokens=200)
    print(r2.choices[0].message.content)
else:
    print(m.content)


---
## Step 6 — Takeaway

*Wrap-up: prompt for cheap, tool for live, deterministic helper for business rules.*
